In [ ]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a],check=True)
pip('diffusers','transformers','accelerate','safetensors','torch','torchvision')
pip('kokoro>=0.9.4','soundfile','numpy')
pip('faster-whisper')
pip('audiocraft')
pip('Pillow','requests')
print('✅ OK')

In [ ]:
import os,json,subprocess
from pathlib import Path

TOPIC    = os.environ.get('TOPIC',   'The Fall of Ancient Rome')
NICHE    = os.environ.get('NICHE',   'business_collapse')
DURATION = int(os.environ.get('DURATION','10'))
LANG     = os.environ.get('LANG',    'en')
OPENROUTER_KEY = os.environ.get('OPENROUTER_API_KEY','')

NICHES = {
    'business_collapse': {
        'voice':       'bm_lewis',
        'music_mood':  'dramatic orchestral cinematic dark',
        'image_style': 'cinematic documentary still, dark moody lighting, cold cyan teal grade, 8K photorealistic',
        'color_grade': 'cold_cyan',
    },
    'ancient_stoicism': {
        'voice':       'am_michael',
        'music_mood':  'ambient philosophical ancient calm meditation',
        'image_style': 'ancient philosophy documentary, marble ruins golden hour, warm soft light, 8K cinematic',
        'color_grade': 'warm_gold',
    },
    'sleep_stories': {
        'voice':       'af_heart',
        'music_mood':  'soft rain ambient sleep relaxing',
        'image_style': 'dreamy night forest moonlight, soft blue midnight atmosphere, watercolor style, 8K',
        'color_grade': 'midnight_blue',
    },
}

preset = NICHES.get(NICHE, NICHES['business_collapse'])
OUT = Path('/kaggle/working/output')
OUT.mkdir(exist_ok=True)
print(f'Topic: {TOPIC} | Niche: {NICHE} | Voice: {preset["voice"]}')

In [ ]:
import requests

def gen_script(topic, minutes, lang='en'):
    words = minutes * 130
    lang_instr = 'Write in English, BBC documentary style, authoritative cinematic tone.' if lang=='en' else 'Write in Russian, BBC style.'
    prompt = f'Write a {minutes}-minute documentary narration about: {topic}. Length: {words} words. {lang_instr} Plain narration only, no headers, no stage directions.'
    models = [
        'google/gemini-2.0-flash-exp:free',
        'meta-llama/llama-3.3-70b-instruct:free',
        'deepseek/deepseek-r1:free',
    ]
    for model in models:
        try:
            r = requests.post(
                'https://openrouter.ai/api/v1/chat/completions',
                headers={'Authorization': f'Bearer {OPENROUTER_KEY}','Content-Type':'application/json'},
                json={'model':model,'messages':[{'role':'user','content':prompt}],'max_tokens':4000,'temperature':0.3},
                timeout=90)
            content = r.json()['choices'][0]['message']['content'].strip()
            if content and len(content) > 200:
                print(f'✅ Script: {model} ({len(content.split())} words)')
                return content
        except Exception as e:
            print(f'⚠️ {model}: {e}')
    raise RuntimeError('❌ Failed to get script')

script = gen_script(TOPIC, DURATION, LANG)
script_path = OUT / 'script.txt'
script_path.write_text(script, encoding='utf-8')
print(f'📄 Script saved: {len(script.split())} words')

In [ ]:
import soundfile as sf
import numpy as np
from kokoro import KPipeline

def gen_narration(script_path, output_path, voice):
    text = script_path.read_text('utf-8')
    lang_code = 'b' if voice.startswith('b') else 'a'
    pipeline = KPipeline(lang_code=lang_code)
    chunks = []
    for _,_,audio in pipeline(text, voice=voice, speed=0.88, split_pattern=r'(?<=[.!?])\s+'):
        chunks.append(audio)
    if not chunks:
        raise RuntimeError('Kokoro empty output')
    combined = np.concatenate(chunks)
    wav = output_path.with_suffix('.wav')
    sf.write(str(wav), combined, 24000)
    subprocess.run(['ffmpeg','-y','-i',str(wav),'-c:a','libmp3lame','-q:a','2',str(output_path)],check=True,capture_output=True)
    wav.unlink(missing_ok=True)
    print(f'✅ Narration: {output_path.stat().st_size//1024} KB')

narration_path = OUT / 'narration.mp3'
gen_narration(script_path, narration_path, preset['voice'])

In [ ]:
from faster_whisper import WhisperModel

def gen_subtitles(audio_path, srt_path, lang='en'):
    model = WhisperModel('large-v3', device='cuda', compute_type='float16')
    segments, _ = model.transcribe(str(audio_path), language=lang, word_timestamps=True, beam_size=5)
    def ts(s):
        h=int(s//3600); m=int((s%3600)//60); sec=int(s%60); ms=int((s%1)*1000)
        return f'{h:02}:{m:02}:{sec:02},{ms:03}'
    lines = []
    for i,seg in enumerate(segments,1):
        lines.append(f'{i}\n{ts(seg.start)} --> {ts(seg.end)}\n{seg.text.strip()}\n')
    srt_path.write_text('\n'.join(lines), encoding='utf-8')
    print(f'✅ Subtitles: {len(lines)} segments')

subtitles_path = OUT / 'subtitles.srt'
gen_subtitles(narration_path, subtitles_path, LANG)

In [ ]:
import torch
from diffusers import FluxPipeline

def gen_images(script, style, n, out_dir):
    out_dir.mkdir(exist_ok=True)
    sentences = [s.strip() for s in script.replace('\n',' ').split('.') if len(s.strip())>20]
    step = max(1, len(sentences)//n)
    scenes = [' '.join(sentences[i*step:(i+1)*step])[:200] for i in range(n)]
    pipe = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell', torch_dtype=torch.float16)
    pipe.enable_sequential_cpu_offload()
    pipe.enable_attention_slicing()
    paths = []
    for i,scene in enumerate(scenes):
        print(f'🎨 [{i+1}/{n}] generating...')
        img = pipe(f'{scene}, {style}', num_inference_steps=4, guidance_scale=0.0, height=1080, width=1920).images[0]
        p = out_dir / f'img_{i:03d}.png'
        img.save(str(p))
        paths.append(p)
    del pipe; torch.cuda.empty_cache()
    return paths

n_images = max(6, DURATION)
image_paths = gen_images(script, preset['image_style'], n_images, OUT/'images')
print(f'✅ {len(image_paths)} images generated')

In [ ]:
import torch
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

narr_dur = int(float(subprocess.check_output(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(narration_path)]).decode().strip()))

def gen_music(mood, duration_sec, output_path):
    model = MusicGen.get_pretrained('facebook/musicgen-small')
    model.set_generation_params(duration=min(duration_sec,30), temperature=0.9, top_k=250)
    wav = model.generate([mood])
    tmp = output_path.with_suffix('.tmp.wav')
    audio_write(str(tmp.with_suffix('')), wav[0].cpu(), model.sample_rate, strategy='loudness')
    subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',str(tmp),'-t',str(duration_sec),'-c:a','libmp3lame','-q:a','4',str(output_path)],check=True,capture_output=True)
    tmp.unlink(missing_ok=True)
    del model; torch.cuda.empty_cache()
    print(f'✅ Music: {output_path.stat().st_size//1024} KB')

music_path = OUT / 'music.mp3'
gen_music(preset['music_mood'], narr_dur, music_path)

In [ ]:
def build_video(image_paths, duration_sec, output_path):
    clip_dur = max(4, duration_sec//len(image_paths))
    clips_dir = OUT/'clips'; clips_dir.mkdir(exist_ok=True)
    movements = [
        "zoompan=z='min(zoom+0.0008,1.5)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={d}:s=1920x1080",
        "zoompan=z=1.3:x='if(lte(on,1),0,x+1.2)':y='ih/2-(ih/zoom/2)':d={d}:s=1920x1080",
        "zoompan=z='if(lte(on,1),1.5,max(1,zoom-0.0008))':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={d}:s=1920x1080",
        "zoompan=z=1.3:x='iw/2-(iw/zoom/2)':y='if(lte(on,1),0,y+0.8)':d={d}:s=1920x1080",
    ]
    clip_paths = []
    for i,img in enumerate(image_paths):
        cp = clips_dir/f'clip_{i:03d}.mp4'
        mv = movements[i%len(movements)].format(d=clip_dur*25)
        subprocess.run(['ffmpeg','-y','-loop','1','-i',str(img),'-vf',f'{mv},scale=1920:1080,setsar=1','-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','20','-pix_fmt','yuv420p','-r','25',str(cp)],check=True,capture_output=True)
        clip_paths.append(cp)
        print(f'  🎬 [{i+1}/{len(image_paths)}]')
    manifest = clips_dir/'list.txt'
    manifest.write_text('\n'.join(f"file '{p.resolve()}'" for p in clip_paths))
    subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(manifest),'-c','copy',str(output_path)],check=True,capture_output=True)
    print(f'✅ Visuals: {output_path.stat().st_size//1024//1024} MB')

visuals_path = OUT/'visuals.mp4'
build_video(image_paths, narr_dur, visuals_path)

In [ ]:
GRADES = {
    'cold_cyan':     'curves=r=0/0 0.2/0.08 0.8/0.62 1/0.82:g=0/0 0.2/0.10 0.8/0.63 1/0.82:b=0/0.06 0.8/0.8 1/1',
    'warm_gold':     'curves=r=0/0 0.5/0.6 1/1:g=0/0 0.5/0.5 1/0.9:b=0/0 0.5/0.3 1/0.7,eq=saturation=1.2',
    'midnight_blue': 'curves=r=0/0 0.5/0.35 1/0.75:g=0/0 0.5/0.4 1/0.8:b=0/0.1 0.5/0.6 1/1,eq=brightness=-0.05',
}

grade = GRADES.get(preset['color_grade'], GRADES['cold_cyan'])
sub_style = 'FontName=Arial,FontSize=22,Bold=1,PrimaryColour=&Hffffff&,OutlineColour=&H000000&,Outline=2,Shadow=1,Alignment=2,MarginV=40'
fade_out = max(0, narr_dur-4)

vf = ','.join([
    'scale=1920:1080:force_original_aspect_ratio=decrease',
    'pad=1920:1080:(ow-iw)/2:(oh-ih)/2',
    grade,
    'noise=alls=4:allf=t+u',
    'vignette=PI/5',
    f"subtitles={subtitles_path}:force_style='{sub_style}'",
])
audio_f = (
    f'[1:a]volume=1.0,afade=t=in:d=0.5,afade=t=out:st={fade_out}:d=4[narr];'
    f'[2:a]volume=0.12,afade=t=in:d=3,afade=t=out:st={fade_out}:d=4[mus];'
    f'[narr][mus]amix=inputs=2:duration=first[audio]'
)

final_path = OUT/'final.mp4'
subprocess.run([
    'ffmpeg','-y',
    '-i',str(visuals_path),'-i',str(narration_path),'-i',str(music_path),
    '-vf',vf,'-filter_complex',audio_f,
    '-map','0:v','-map','[audio]',
    '-c:v','libx264','-profile:v','high','-level:v','4.0',
    '-preset','medium','-crf','18','-r','25',
    '-c:a','aac','-b:a','192k','-ar','44100',
    '-pix_fmt','yuv420p','-movflags','+faststart','-shortest',
    str(final_path)
],check=True)

size = final_path.stat().st_size//1024//1024
dur2 = int(float(subprocess.check_output(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(final_path)]).decode().strip()))
print(f'\n✅ ГОТОВО: {final_path}')
print(f'   Длина: {dur2//60}:{dur2%60:02d} | Размер: {size} MB')